# 12 — Полный цикл обучения CrafText: rollout → replay → trainable token PPO update

Эта тетрадка связывает текущий synchronous CrafText pipeline в один обучающий цикл: batched CrafText rollout, сохранение replay evidence, преобразование replay в `TextTrajectoryBatch`, расчёт returns и один full-token trainable actor-critic PPO update, где все generated tokens входят в loss/update. Она намеренно маленькая: цель — доказать контракты данных, маски, actor logprob recomputation и critic values, а не обучить production Qwen/RLCluster workload.


In [ ]:
from pathlib import Path
import jax
import jax.numpy as jnp
from tunix_craftext.algorithms import masked_token_returns
from tunix_craftext.batched_rollout import collect_batched_text_rollout, replays_from_batched_rollout
from tunix_craftext.config import load_mvp_config
from tunix_craftext.learner import create_token_state, full_token_ppo_update, token_actor_critic_outputs, token_ppo_update
from tunix_craftext.llm import LlmResponse
from tunix_craftext.prompts import MegaPromptRenderer
from tunix_craftext.replay import save_replay
from tunix_craftext.runtime import build_craftext_runtime
from tunix_craftext.text_trajectory import text_trajectory_from_replay


In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook inside tunix-craftext.')
config = load_mvp_config(ROOT / 'configs/mvp/qwen_craftext.yaml')
runtime = build_craftext_runtime(config)
print('CrafText actions:', runtime.actions.labels)

In [ ]:
class ScriptedTrainingBackend:
    '''Deterministic stand-in for a batched Qwen sampler with token provenance.'''
    def __init__(self):
        self.calls = 0
    def complete(self, request):
        return self.complete_batch((request,))[0]
    def complete_batch(self, requests):
        self.calls += 1
        responses = []
        for row, _ in enumerate(requests):
            # Alternate valid CrafText labels while retaining token/logprob evidence.
            label = runtime.actions.labels[(self.calls + row) % len(runtime.actions.labels)]
            responses.append(LlmResponse(
                raw_text=f'<action>{label}</action>',
                model='scripted-training-backend',
                backend='notebook',
                token_ids=(100 + self.calls, 200 + row),
                token_logprobs=(-0.4, -0.2),
                prompt_token_ids=(1, 2, 3, 4),
            ))
        return tuple(responses)

backend = ScriptedTrainingBackend()

In [ ]:
rollout = collect_batched_text_rollout(
    runtime.adapter,
    MegaPromptRenderer(config.prompt.template),
    backend,
    actions=runtime.actions,
    batch_size=8,
    horizon=150,
    seed=config.run.seed,
    goal='Collect useful CrafText experience for a tiny PPO update.',
    max_new_tokens=8,
    invalid_action='fallback',
    fallback_action_id=runtime.actions.index_of('NOOP'),
)
print('LLM batch calls:', backend.calls)
for t, decision in enumerate(rollout.decisions):
    print('t=', t, 'actions=', [action.label for action in decision.actions], 'reward=', decision.transition.reward.tolist())

In [ ]:
import jax
print(jax.default_backend())

In [ ]:
output_dir = ROOT / 'artifacts/trajectories/full-cycle-craftext-training'
replays = replays_from_batched_rollout(rollout, config_path='configs/mvp/qwen_craftext.yaml', commit='notebook', backend='scripted')
for env_index, replay in enumerate(replays):
    save_replay(output_dir / f'env-{env_index}.json', replay)
    print('saved replay', env_index, 'steps=', len(replay.steps))

In [ ]:
batches = tuple(text_trajectory_from_replay(replay) for replay in replays)
batch = batches[0]
returns = masked_token_returns(batch.rewards, batch.token_mask, gamma=0.99)
old_actor_state = create_token_state(
    jax.random.PRNGKey(42),
    token_bucket_count=128,
    hidden=32,
    learning_rate=3e-4,
)
old_new_logprobs, old_values, old_entropy = token_actor_critic_outputs(old_actor_state, batch)
actor_state, metrics = full_token_ppo_update(old_actor_state, batch, gamma=0.99)
new_new_logprobs, new_values, _ = token_actor_critic_outputs(actor_state, batch)
print('token_ids:', batch.token_ids.tolist())
print('token_mask:', batch.token_mask.tolist())
print('policy_mask:', batch.policy_mask.tolist(), '(safe PPO would exclude fallback tokens)')
print('returns:', returns.tolist())
print('old trainable logprobs:', old_new_logprobs.tolist())
print('old critic values:', old_values.tolist())
print('new critic values:', new_values.tolist())
print('full-token PPO loss:', float(metrics['loss']))
print({name: float(value) for name, value in metrics.items()})
print('learned tokens:', float(metrics['learned_tokens']))
print('actor changed:', bool(jnp.any(jnp.abs(new_new_logprobs - old_new_logprobs) > 1e-7)))


In [ ]:
replay = replays[0]
list(map(lambda x: x, replay.steps))

## Что заменяется в production loop

- `ScriptedTrainingBackend` заменяется на `QwenTunixBackend` или future `RLCluster.ROLLOUT` actor.
- Compact `PromptConditionedTokenActorCritic` заменяется на production actor/value path: Qwen actor пересчитывает token logprobs, critic/value head выдаёт values.
- Notebook использует `full_token_ppo_update()`: все generated tokens из `token_mask` входят в `masked_token_ppo_loss`; padding исключён, fallback tokens не выкидываются.
- Safe policy-only режим остаётся как `token_ppo_update()` для запусков, где fallback не должен становиться обучающим сигналом.
- Notebook уже сохраняет replay evidence до loss, чтобы audit мог проверить prompt, action, token ids/logprobs, rewards и fallback masks.
- Следующий hard gate — hardware-gated RLCluster rollout/update с checkpoint actor/critic/optimizer/policy version.
